In [0]:
# import libraries

from pyspark.sql import SparkSession, functions as F

from pyspark.sql.functions import (
    explode, desc,  row_number, col, try_divide, year, try_to_date, count, 
    countDistinct
)

from pyspark.sql.window import Window

# import pandas as pd


In [0]:
# curl https://full-stack-bigdata-datasets.s3.amazonaws.com/Big_Data/Project_Steam/steam_game_output.json
# record a file in local wget -O steam_game_output.json https://full-stack-bigdata-datasets.s3.amazonaws.com/Big_Data/Project_Steam/steam_game_output.json

In [0]:
# from pyspark.sql import SparkSession

filepath = "s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json"

df = spark.read.format('json').load(filepath)

In [0]:
# Number of elements in dataframe
print(f"Entries number : {df.count()}")


The dataset is to big for json_normalize method.

In [0]:
type(df)

In [0]:
df.take(1)

In [0]:
df.printSchema()

We observe 23 different values : data is a only a node, categories, platforms and tags contain nested informations. Let's flatened the dataframe to range datas at same level. 

In [0]:
from pyspark.sql.functions import concat_ws

flat_df = df.select("id","data.*", concat_ws(", ", "data.categories").alias("category"))
flat_df.limit(1).display()

In [0]:
# from pyspark.sql import functions as F

flat_df = flat_df.withColumn(
    "platform",
    F.concat_ws(', ', *[F.when(F.col(f"platforms.{f}"), F.lit(f)) for f in ['linux', 'mac', 'windows']])
)
flat_df.limit(1).display()

In [0]:
flat_df = flat_df.withColumn("tag", F.to_json(F.col("tags")))
flat_df.limit(1).display()

In [0]:
flat_df = flat_df.drop( "categories", "platforms", "tags")

In [0]:
flat_df.printSchema()

In [0]:
# flat_df.write.csv("df.csv", header=True)

Export a csv with the .write method requires missing write right on working directory in databricks free edition. So, let's use pandas method .tocsv()

In [0]:
# import pandas as pd

# flat_df.toPandas().to_csv("df.csv", index=False)

# Explorary dataset analysis

## 1. Explore dataset

In [0]:
len(flat_df.columns)

In [0]:
flat_df.count()

The dataframe has 23 columns and 55691 rows.

Now, let's query on the dataframe. Spark lets us run classic SQL queries on your tables, however, using classic SQL in Spark requires you to load the data in memory before running any query. We will use the .createOrReplaceTempView Spark DataFrame method in order to load the data in memory under a certain table name, we will then be able to run SQL queries on it.

In [0]:
flat_df.createOrReplaceTempView('temp_table') # Creates a temporary view table in memory called temp_table
result = spark.sql("select * from temp_table limit 1")
result.show()

The .sql method lets you write queries in SQL while benefiting from the distributed computing advantages of Spark.

In [0]:
result = spark.sql("SELECT * FROM temp_table LIMIT 1") # filters elements from temp_table 
# show everything
result.show()

We observe 
- ccu nombre de joueurs qui jouaient simulatanément au moment où le dataset a été construit
- discount en pourcentage
- prix stocké en centimes de dollars
- owners indique un intervalle pour évaluer le nombre d'acheteurs
- categories la liste varie selon le jeu. Elles permettent aux acheteurs de classer les jeux et fonctionnent comme les étagères d'une bibliothèque.
- duplicated rows ?

some problems :
- unused spaces : sort columns to verify wether it's a display problem or requests problem 
- different spelling : lowercase string columns
- special alphabetic characters on string columns developers '---', "flyingcubicle, - ", "((no-end-parens studio", "revday studio", "+7 software" , "+mpact games, llc."  
- empty entries : '' on website and platform -> verify other columns
- unused columns : id and appid are identifyers, we use name and can delete them, website, header_image 
- des colonnes numériques au format string -> convertir price, initialprice et discount au format long
- release_date string for datetime

We must verify with steam team before treat these data.
- special alphabetic characters : "," is a separator that we can use to count entries on developers, genre, languages. 
Are " ", None, ., CD PROJEKT RED, [2.21] real publishers and '---', "flyingcubicle, - ", "((no-end-parens studio", "revday studio", "+7 software" , "+mpact games, llc." real developers ?
Are "(none)", " ", "-" real publishers ?
- asiatic characters. We need information even if the langage changes.

Let's sample and observe values on selected columns.

In [0]:
flat_df.select('developer', 'publisher', 'owners', 'ccu', 'website', 'header_image').sample(fraction=0.0001).distinct().show(truncate=False)

In [0]:
spark.sql("select * from temp_table limit 5").show()

In [0]:
result = spark.sql("select distinct type from temp_table")
result.show()

In [0]:
result = spark.sql("select * from temp_table where type = 'hardware'")
result.show(truncate=False)

Steam link est un outil édité par Anima Locus qui permet d'accéder à un jeu par lien sur le téléphone, une tablette, un autre PC et même d'accéder à un jeu hébergé sur le pc d'un ami. Ce n'est pas un jeu en soi. Il convient de supprimer la colonne type.

In [0]:
result = spark.sql("select distinct name from temp_table where name like'Counter-Strike%'")
result.show(truncate=False)

In [0]:
from pyspark.sql.functions import col

string_cols = [field.name for field in flat_df.schema.fields if field.dataType.simpleString() == 'string']

for c in string_cols:
    count = flat_df.filter(col(c) == '').count()
    if count > 0:
        print(f"{c}: {count}")

In [0]:
result = spark.sql("select * from temp_table where release_date = ''")
result.show()

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

all_cols = flat_df.columns
string_cols = [f.name for f in flat_df.schema.fields if f.dataType.simpleString() == 'string']

flat_df.select([spark_sum((col(c).isNull() | ((col(c) == '') if c in string_cols else False)).cast('int')).alias(c) for c in all_cols]).show()

There are 55691 rows. Website misses on one half rows. Other missing values should be converted.


In [0]:
flat_df.count() == flat_df.dropDuplicates().count()

In [0]:
(flat_df.count() - flat_df.dropDuplicates().count())

There is no duplicated value.

## 2. Clean data

Let's record clean_data under a new data_frame and create a new temp view.

In [0]:
clean_df = flat_df
clean_df.createOrReplaceTempView('table')

### a) treatments

Let's clean data. First, delete unused spaces with trim method.

In [0]:
clean_df = spark.sql("SELECT * FROM table LIMIT 1")
clean_df.show()

In [0]:
from pyspark.sql.functions import trim

clean_df = spark.sql("SELECT * FROM table") \
    .select([trim(col(c)).alias(c) for c in spark.sql("SELECT * FROM table").columns])
clean_df.show()


Then, convert numerical columns ccu, price, initialprice and discount from string to long.

In [0]:
from pyspark.sql.functions import col

string_cols = ['ccu', 'price', 'initialprice', 'discount']

for c in string_cols:
    clean_df = clean_df.withColumn(c, col(c).cast('long'))


Then, convert release_date from string to datetime

to_date() ne passe pas en raison de valeurs non conformes. Comptez le nombre de segments (/) pour chaque valeur distincte de 'release_date', afin de voir toutes les structures présentes (3 segments, 2 segments, etc.).

In [0]:
from pyspark.sql.functions import col, split, size

clean_df.select('release_date', size(split(col('release_date'), '/')).alias('partition_nb')) \
    .distinct() \
    .groupBy('partition_nb') \
    .count() \
    .show()

Résultat clair: Trois structures existent dans 'release_date':

3 segments (3937 lignes): format complet 'yyyy/M/d'
2 segments (73 lignes): année/mois seulement, ex '2019/01'
1 segment (1 ligne): probablement juste l'année

In [0]:
from pyspark.sql.functions import to_date, col, when, split, size, concat, lit

nb_segments = size(split(col('release_date'), '/'))

clean_df = clean_df.withColumn(
    'release_date',
    when(col('release_date') == '', None)
    .when(nb_segments == 3, to_date(col('release_date'), 'yyyy/M/d'))
    .when(nb_segments == 2, to_date(concat(col('release_date'), lit('/1')), 'yyyy/M/d'))
    .when(nb_segments == 1, to_date(concat(col('release_date'), lit('/1/1')), 'yyyy/M/d'))
    .otherwise(None)
)
clean_df.select('release_date').show(5, truncate=False)

In [0]:
price_cols = [ 'price', 'initialprice']

for c in price_cols:
    clean_df = clean_df.withColumn(c, col(c) / 100)
clean_df.select('price', 'initialprice').show(5)


Then, deal with latest missing values.

In [0]:
from pyspark.sql.functions import col, when, to_date, lit

for field in clean_df.schema.fields:
    c = field.name
    if field.dataType.simpleString() == 'string':
        clean_df = clean_df.withColumn(c, when(col(c) == '', 'None').otherwise(col(c)))
    elif c == 'release_date':
        clean_df = clean_df.withColumn(c, when(col(c).isNull(), to_date(lit('1900-01-01'), 'yyyy-MM-dd')).otherwise(col(c)))

clean_df.filter(col('release_date') == to_date(lit('1900-01-01'), 'yyyy-MM-dd')).show(1)

In [0]:
clean_df.filter(col('release_date').isNull()).count()

Then, lowercase all values.

In [0]:
from pyspark.sql.functions import lower, col

string_cols = [f.name for f in clean_df.schema.fields if f.dataType.simpleString() == 'string']

clean_df = clean_df.select([lower(col(c)).alias(c) for c in string_cols] + [col(c) for c in clean_df.columns if c not in string_cols])
clean_df.show(1)

Endly, drop unused columns.

In [0]:
clean_df = clean_df.drop('id', 'type', 'header_image', 'website')

### b) verify clean_df dataframe

In [0]:
clean_df.createOrReplaceTempView('table') # Refresh the temporary view table in memory
result = spark.sql("select * from table limit 1")
result.show()

In [0]:
clean_df.printSchema()

In [0]:
# Number of elements in dataframe
print(f"Entries number : {clean_df.count()}")

In [0]:
result = spark.sql("select * from table limit(5)")
result.show()

In [0]:
clean_df.select('name', 'release_date', 'developer', 'publisher', 'owners', 'ccu').sample(fraction=0.0001).show(truncate=False)

Verify missing values.

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

all_cols = clean_df.columns
string_cols = [f.name for f in clean_df.schema.fields if f.dataType.simpleString() == 'string']

clean_df.select([spark_sum((col(c).isNull() | ((col(c) == '') if c in string_cols else False)).cast('int')).alias(c) for c in all_cols]).show()

In [0]:
result = spark.sql("select * from table where developer = 'none' or genre = 'none' or languages = 'none' or publisher = 'none' or short_description = 'none' or category = 'none'")
result.limit(1).show()

In [0]:
clean_df.filter(col('release_date').isNull()).count()

In [0]:
result = clean_df \
    .filter(col('release_date') == to_date(lit('1900-01-01'), 'yyyy-MM-dd'))
result.limit(1).show()

In [0]:

clean_df.describe().toPandas()

In [0]:
result = clean_df \
    .filter(col('release_date') != to_date(lit('1900-01-01'), 'yyyy-MM-dd')) \
    .select( \
        F.min("release_date").alias("min_release_date"),
        F.max("release_date").alias("max_release_date"))
result.show()

# TO DO Nico graph time series

## 3. Analysis at the "macro" level

Which publisher has released the most games on Steam?

In [0]:
from pyspark.sql.functions import count, desc

result = clean_df \
    .select(clean_df['publisher'],'name', 'release_date') \
    .distinct() \
    .groupBy('publisher') \
    .agg(
        count('name').alias('game_nb'),
        countDistinct('release_date').alias('release_nb')
        ) \
    .orderBy(desc('release_nb')) \
    .limit(1)
result.show()

What is Valve's position as publisher ?

In [0]:
from pyspark.sql.functions import count, desc

result = clean_df \
    .select(clean_df['publisher'],'name', 'release_date') \
    .groupBy('publisher') \
    .agg( \
        count('name').alias('game_nb'), \
        countDistinct('release_date').alias('release_nb')) \
    .withColumn('rank', row_number().over(Window
    .orderBy(desc('release_nb')))) \
    .filter(clean_df.publisher.isin(['big fish games','valve']))
result.show()

How many games on steam ?

In [0]:
result = clean_df \
    .select(clean_df['name']) \
    .distinct() \
    .count() 
print(f"Games number : {result}")

How many developers indicated on steam ?

In [0]:
result = clean_df \
    .select(clean_df['developer']) \
    .distinct() \
    .count()
print(f"Developer number : {result}")



Which developer most contribute to games on steam ?

In [0]:
result = clean_df \
    .select(clean_df['developer'],'name') \
    .distinct() \
    .groupBy('developer') \
    .agg(count('name').alias('game_nb')) \
    .orderBy(desc('game_nb')) \
    .limit(5)
result.show(truncate=False)

What are the best rated games?

In [0]:
result = clean_df \
    .select(clean_df['name'],'positive') \
    .groupBy('name') \
    .agg(F.sum('positive').alias('positive_nb')) \
    .orderBy(desc('positive_nb')) \
    .limit(10)
result.show()

In [0]:
from pyspark.sql.functions import col, try_divide

result = clean_df \
    .select(clean_df['name'],'positive', 'negative') \
    .groupBy('name') \
    .agg( \
        F.sum('positive').alias('positive_sum'), \
        F.sum('negative').alias('negative_sum') \
        ) \
    .filter((col('positive_sum') > 0) & (col('negative_sum') > 0)) \
    .withColumn('ratio', col('positive_sum') /  col('negative_sum')) \
    .orderBy(desc('positive_sum')) \
    .limit(10)
result.show(truncate=False)

In absolute value, Counter-Strike: Global Offensive has more positive advices (65,4M). Yet Terraria has a better ratio : 45,34 positive advices for 1 negative advice against 7,5 on Counter-Strike. Then, Terraria is proportianaly more liked.

Quels sont les jeux les plus utilisés au moment où le dataset a été construit ?

In [0]:
from pyspark.sql.functions import col, try_divide

result = clean_df \
    .select(clean_df['name'],'ccu') \
    .orderBy(desc('ccu'))
result.show(truncate=False)

Et quels sont les jeux les plus achetés ?

In [0]:
from pyspark.sql.functions import col, try_divide

result = clean_df \
    .groupBy('owners') \
    .agg( \
        F.count('owners').alias('owners_sum')) \
    .orderBy('owners_sum')
result.show(truncate=False)

In [0]:
from pyspark.sql.functions import col, regexp_replace, split, desc

result = clean_df \
    .select('name', 'owners', 'release_date') \
    .withColumn('owners_num', regexp_replace(split(col('owners'), ' ')[0], ',', '').cast('long')) \
    .orderBy(desc('owners_num')) \
    .drop('owners_num') \
    .limit(10)
result.show(truncate=False)

Les jeux les plus plébiscités ne sont pas forcément les plus utilisés ou les plus achetés. counter-strike: global offensive était le jeu le plus utilisé avec 874053 utilisateurs. Et dota 2 était le jeu le plus vendu avec 200 000 000 à 500 000 000 utilisateurs. counter-strike: global offensive n'a que 50 000 000 à 100 000 000 d'acheteurs. Avec la release_date, on s'aperçoit que data 2 et conter-strike offensive sont anciens.

In [0]:
result = clean_df.groupBy('owners').agg(F.count('*').alias('count')).orderBy(desc('count'))
result.show(truncate=False)

Are there years with more releases? Were there more or fewer game releases during the Covid, for example?

In [0]:
from pyspark.sql.functions import (
    col, year, try_to_date, count, countDistinct
)

result = (flat_df \
    .select("name", "release_date") \
    .distinct() \
    .withColumn( \
        "release_year", \
        year(try_to_date(col("release_date"), "yyyy/M/d")) \
    ) \
    .filter(col("release_year").isNotNull()) \
    .groupBy("release_year") \
    .agg( \
        count("*").alias("release_nb"), \
        countDistinct("name").alias("name_nb") \
    ) \
    .filter((col('release_nb') > 0) & (col('name_nb') > 0)) \
    .withColumn('ratio', col('release_nb') /  col('name_nb')) \
    .orderBy(desc("release_year")) \
)

result.show()

Yes. There are years with more releases. 
- In absolute value, 2014 to 2022. There are more releases after the covid's year in 2021 with 8676. 
- In prortional value, 2015 to 2022. Releases become higher than games since 2015. Yet, releases are higher on 2020, the covid's year with 1.0011 releases for one game.  
We observe that Covid accentuated the trend which returns then at normal pace in 2022 with 7401 and 1.00094.

In [0]:
result = spark.sql("SELECT * FROM table") # filters elements from my_table where position
# show everything
result.show()

# TO DO Nico graph avec quartiles ? time series ?
How are the prizes distributed? Are there many games with a discount? regarder avec et sans mettre un count

In [0]:
q1, q3 = clean_df.approxQuantile(
    "price",
    [0.25, 0.75],
    0.001
)

iqr = q3 - q1

upper_bound = q3 + 1.5 * iqr

print(f"Q1 : {q1:.2f} $")
print(f"Q3 : {q3:.2f} $")
print(f"IQR : {iqr:.2f} $")
print(f"Upper bound : {upper_bound:.2f} $")

In [0]:
price_df = (
    clean_df
    .select("name", "price")
    .filter(F.col("price").isNotNull())
    .withColumn(
        "price_status",
        F.when(
            F.col("price") > upper_bound,
            "Outlier statistique"
        ).otherwise("Prix courant")
    )
)

In [0]:
median_price = clean_df.approxQuantile(
    "price",
    [0.5],
    0.001
)[0]

print(f"Médiane du prix : {median_price:.2f} $")

In [0]:
display(
    clean_df
    .select("price")
    .filter(F.col("price").isNotNull())
    .withColumn(
    "price_status",
    F.when(
        F.col("price") > upper_bound,
        "Outlier statistique"
    ).otherwise("Prix courant")
))

Databricks visualization. Run in Databricks to view.

In [0]:
bin_width = 1 # La granularité à 1 est plus pertinente qu'a 2 pour éviter d'avoir le seuil d'outlier dans une classe. 

In [0]:
# 3. Distribution des jeux par classe de prix
result_bin_1 = (
    clean_df
    .filter(F.col("price").isNotNull())

    # Classe de prix : 0, 2, 4, 6, 8...
    .withColumn(
        "price_bin",
        F.floor(F.col("price") / bin_width) * bin_width
    )


    # Séparation prix courants / outliers
    .withColumn(
        "price_status",
        F.when(
            F.col("price") > upper_bound,
            "Outlier statistique"
        ).otherwise("Prix courant")
    )

    # Comptage des jeux
    .groupBy("price_bin", "price_status", "owners")
    .agg(
        F.count("*").alias("game_nb")
    )

    .orderBy("price_bin")
)

display(result)

Databricks visualization. Run in Databricks to view.

In [0]:
bin_width = 5 # Nous modifions la granularité des prix après drop des outliers qu'on ne garde pas afin de faire apparaitre les tranches de prix les plus utilisées. 

# 3. Distribution des jeux par classe de prix
result = (
    clean_df
    .filter(F.col("price").isNotNull())
    .drop(F.col("price") > 60)
    # Classe de prix : 0, 2, 4, 6, 8...
    .withColumn(
        "price_bin",
        F.floor(F.col("price") / bin_width) * bin_width
    )


    # Séparation prix courants / outliers
    .withColumn(
        "price_status",
        F.when(
            F.col("price") > upper_bound,
            "Outlier statistique"
        ).otherwise("Prix courant")
    )

    # Comptage des jeux
    .groupBy("price_bin", "price_status")
    .agg(
        F.count("*").alias("game_nb")
    )

    .orderBy("price_bin")
)

display(result)

In [0]:
from pyspark.sql import functions as F

bin_width = 5

price_owners = (
    clean_df

    # 1. Transformation de owners :
    # "20,000 .. 50,000" -> min = 20000 / max = 50000
    .withColumn(
        "owners_min",
        F.regexp_replace(
            F.trim(F.split(F.col("owners"), r"\.\.").getItem(0)),
            ",",
            ""
        ).cast("long")
    )
    .withColumn(
        "owners_max",
        F.regexp_replace(
            F.trim(F.split(F.col("owners"), r"\.\.").getItem(1)),
            ",",
            ""
        ).cast("long")
    )

    # 2. Estimation du nombre d'owners du jeu
    # milieu de l'intervalle
    .withColumn(
        "owners_mean",
        (
            ((F.col("owners_min") + F.col("owners_max")) / 2)
        ).cast("long")
    )

    # 3. Création des tranches de prix
    # 0-5$, 5-10$, 10-15$...
    .withColumn(
        "price_bin",
        F.floor(F.col("price") / bin_width) * bin_width
    )

    # 4. Agrégation par tranche de prix
    .groupBy("price_bin")
    .agg(
        F.count("*").alias("game_nb"),
        F.sum("owners_mean").alias("total_owners"),
        F.avg("owners_mean").alias("mean_owners")
    )

    # 5. Coefficient :
    # nombre moyen d'owners par jeu dans la tranche
    .withColumn(
        "coef",
        F.col("total_owners") / F.col("game_nb") * 0.0001
    )

    .orderBy("price_bin")
)

display(price_owners)

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql import functions as F

bin_width = 5

price_owners = (
    clean_df

    # 1. Extraction des bornes min / max depuis la string owners
    .withColumn(
        "owners_min",
        F.regexp_replace(
            F.trim(F.split(F.col("owners"), r"\.\.").getItem(0)),
            ",",
            ""
        ).cast("long")
    )
    .withColumn(
        "owners_max",
        F.regexp_replace(
            F.trim(F.split(F.col("owners"), r"\.\.").getItem(1)),
            ",",
            ""
        ).cast("long")
    )

    # 2. Estimation du nombre d'owners par jeu
    .withColumn(
        "owners_mean",
        (
            (F.col("owners_min") + F.col("owners_max")) / 2
        ).cast("long")
    )

    # 3. Création des tranches de prix
    .withColumn(
        "price_bin",
        F.floor(F.col("price") / bin_width) * bin_width
    )

    # 4. Agrégation par tranche de prix
    .groupBy("price_bin")
    .agg(
        F.count("*").alias("game_nb"),
        F.sum("owners_mean").alias("total_owners"),
        F.avg("owners_mean").alias("mean_owners")
    )

    # 5. Coefficient logarithmique
    .withColumn(
        "coef_log",
        F.log10(F.col("mean_owners") + 1)
    )

    .orderBy("price_bin")
)

display(price_owners)

Databricks visualization. Run in Databricks to view.

In [0]:
# création d'un coeficient issu du rapport nb_games / nb_owners
game_nb = result.select("game_nb").first()["game_nb"]
mean_owners = owners_df.select("mean_owners").first()["mean_owners"]

coef_df = spark.createDataFrame(
    [(game_nb, mean_owners, game_nb / mean_owners)],
    ["game_nb", "mean_owners", "coef"]
)

display(coef_df)

In [0]:
outlier_games = (
    price_df
    .filter(F.col("price") > upper_bound)
    .select("name", "price")
    .orderBy(F.desc("price"))
)

display(outlier_games)

In [0]:
clean_df.printSchema()

In [0]:
result = clean_df \
    .groupBy('price') \
    .agg( \
        F.count('price').alias('game_nb')) \
    .orderBy('price')
display(result)

Databricks visualization. Run in Databricks to view.

Prices varies on a scale from 0 to 999 $, sometime of any cents. Mean price is 7,73 $ with an initial price higher on 7,93 $ (cf last run clean_df.describe() command) (cf last run clean_df.describe() command). 

In [0]:
result = clean_df \
    .groupBy(
        (F.floor(F.col('price') / 10) * 10).alias('range_price')) \
    .agg( \
        F.count('price').alias('game_nb')) \
    .orderBy('range_price')
display(result)

Databricks visualization. Run in Databricks to view.

Prices don't change ? 

In [0]:
result = clean_df \
    .select(clean_df['publisher'], 'name', 'release_date', 'price', 'initialprice', 'discount') \
    .filter((clean_df['price'] != clean_df['initialprice']) & (clean_df['discount'] == 0)) \
    .orderBy('publisher', 'name', 'release_date') \
    .show(truncate=False)

Prices change only with discount.

Prices varies relying to discount. Le discount varie de 0 à 90 avec une valeur moyenne de 2,6. Le discount est rare.

In [0]:
result = clean_df \
    .select(clean_df['publisher'], 'developer', 'name', 'owners', 'positive', 'negative', 'release_date', 'price', 'initialprice', 'discount', 'genre', 'languages', 'required_age', 'category', 'platform', ) \
    .filter(clean_df.discount != 0 ) \
    .orderBy(desc('discount')) \
    .show()

Vérifier la répartition des jeux discount

What are the most represented languages?

Are there many games prohibited for children under 16/18?